In [1]:
# Setting up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

In [2]:
from copy import deepcopy
import numpy as np
import gymnasium as gym
from Input_Output_Rxn_Networks.IOCRN_MassAction import IOCRN_MassAction

In [3]:
# Construct a basic CRN
species_labels = ['X_1', 'X_2', 'Z_1', 'Z_2']
inputs_labels = ['u']
stoichiometry_reactants = np.array([[0], [0], [1], [0]], dtype=np.int8)
stoichiometry_products = np.array([[1], [0], [1], [0]], dtype=np.int8)
parameters = np.array([1], dtype=np.float32)
input_influence_matrix = np.array([[0]], dtype=np.int8)
outputs = np.array([2], dtype=np.int8)
IOCRN_AIF = IOCRN_MassAction(stoichiometry_reactants, stoichiometry_products, parameters, input_influence_matrix, outputs, species_labels, inputs_labels)
print('Initial CRN:')
IOCRN_AIF.print_reactions()

Initial CRN:
Inputs: ['u']
Species: ['X_1', 'X_2', 'Z_1', 'Z_2']
Output Species: ['X_2']
Reaction 0: Z_1 -> X_1 + Z_1 ; Rate Constant: 1.0


In [4]:
class CRNEnv(gym.Env):
    """
    Custom Environment that follows gym interface
    This is the basic environment for CRNs.
    """
    def __init__(self, CRN_template, max_num_reactions):
        super(CRNEnv, self).__init__()
        self.CRN_template = CRN_template
        self.action_space = gym.spaces.Dict({
            'reactants space': gym.spaces.Discrete(self.CRN_template.get_complexes_range()),
            'products space': gym.spaces.Discrete(self.CRN_template.get_complexes_range()),
            'input influence space': gym.spaces.Discrete(self.CRN_template.num_inputs + 1),
            'rate constant space': gym.spaces.Box(low=0.0, high=np.inf, shape=(1,), dtype=np.float32)
        })

    def reset(self):
        self.state = deepcopy(self.CRN_template)

    def step(self, action):
        if self.state.num_unknown_parameters > 0:
            self.state.set_next_unknown_parameter(action)
        else:
            self.state.add_reaction(action)
        
        done = False  # Define when the episode ends
        info = {}  # Additional info
        
        return self.state, 0, done, info

    def render(self, mode='human'):
        if mode == 'human':
            self.state.print_reactions()
        else:
            pass  # Implement rendering if needed

In [5]:
class RadomAgent:
    def __init__(self, env, max_rate_constant=10):
        self.env = env
        self.max_rate_constant = max_rate_constant

    def act(self):
        action = {}
        action['reactants index'] = np.random.randint(self.env.action_space['reactants space'].n)
        action['products index'] = np.random.randint(self.env.action_space['products space'].n)
        action['input influence index'] = np.random.randint(self.env.action_space['input influence space'].n)
        action['rate constant'] = np.random.uniform(0, self.max_rate_constant)
        return action

In [6]:
# Construct a template CRN
species_labels = ['X_1', 'X_2', 'Z_1', 'Z_2']
inputs_labels = ['u']
stoichiometry_reactants = np.array([[0], [0], [1], [0]], dtype=np.int8)
stoichiometry_products = np.array([[1], [0], [1], [0]], dtype=np.int8)
parameters = np.array([1], dtype=np.float32)
input_influence_matrix = np.array([[0]], dtype=np.int8)
outputs = np.array([2], dtype=np.int8)
IOCRN_AIF = IOCRN_MassAction(stoichiometry_reactants, stoichiometry_products, parameters, input_influence_matrix, outputs, species_labels, inputs_labels)
TestEnv = CRNEnv(IOCRN_AIF, 10)
agent = RadomAgent(TestEnv)
# Test the environment and agent    
TestEnv.reset()
for i in range(5):
    action = agent.act()
    state, reward, done, info = TestEnv.step(action)
    if done:
        break
TestEnv.render()

Inputs: ['u']
Species: ['X_1', 'X_2', 'Z_1', 'Z_2']
Output Species: ['X_2']
Reaction 0: Z_1 -> X_1 + Z_1 ; Rate Constant: 1.0
Reaction 1: 2 X_2 -> X_1 + Z_2 ; Rate Constant: 0.48393517800630503
Reaction 2: X_2 + Z_2 -> X_1 + Z_2 ; Rate Constant: 9.79122998479842
Reaction 3: X_2 + Z_2 -> X_1 + Z_2 ; Rate Constant: 3.2195510138025796
Reaction 4: 2 X_2 -> 2 X_1 ; Rate Constant: 1.7388699816345887
Reaction 5: Z_1 + Z_2 -> X_2 ; Rate Constant: 6.5574629902933825
